In [1]:
# Standard Library
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from datetime import datetime
from pathlib import Path

# Sklearn Library
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,

)

# Wandb Library
import wandb

# Joblib Library
import joblib as jb

In [2]:
# Khai báo hằng số
RANDOM_STATE = 42
THRESHOLD = 0.4

In [3]:
# Cố định random state để kết quả mỗi lần chạy là như nhau
np.random.seed(RANDOM_STATE)
os.environ['PYTHONHASHSEED'] = '42'

In [4]:
# Cấu hình của Wandb
WANDB_PROJECT = "heart-disease-classification"
WANDB_ENTITY  = "phat-05"

In [5]:
# Thiết lập style cho biểu đồ
sns.set_theme(style="whitegrid", context="notebook")
sns.set_palette('Set2')

In [6]:
# Định nghĩa đường dẫn tới file dataset
dataset_path = Path().resolve().parent / "data/dataset.csv"

# Load data
df = pd.read_csv(dataset_path)

In [7]:
# Thông tin số dòng số cột
print(f"- Dataset có {df.shape[0]} dòng và {df.shape[1]} cột.")

In [8]:
#Thông tin kiểu dữ liệu từng cột, số giá trị non-null
df.info()

In [9]:
#Thông tin về trung bình, độ lệch chuẩn, tứ phân vị, giá trị nhỏ nhất lớn nhất
print("- Các thông tin thống kê numerical:")
display(df.describe())

print("- Các thông tin thống kê categorical:")
display(df.describe(include=['object', 'string']))

In [10]:
# Xem 5 hàng dữ liệu đầu tiên
df.head()

   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  
0       fixed defect    0  
1             normal    2  
2  reversable defect    1  
3             normal    0  
4             normal    0  

In [11]:
# Các cột numerical
num_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']

# Các cột categorical
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

In [12]:
# Dùng hàm astype để convert tất cả dữ liệu trong cột từ bool thành int
df['target'] = (df['num'] > 0).astype(int)

#Hiển thị dữ liệu đã thêm target
df.head()

   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  target  
0       fixed defect    0       0  
1             normal    2       1  
2  reversable defect    1       1  
3             normal    0       

In [13]:
missing_summary = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum(),
    'missing_percent': df.isnull().mean() * 100
}).sort_values(by='missing_percent', ascending=False)

print(missing_summary)

In [14]:
plt.figure(figsize=(10,5))
sns.barplot(x='missing_percent', y='column', data=missing_summary)
plt.title('Tỷ lệ missing theo từng cột (%)')
plt.xlabel('Percent')
plt.ylabel('Column')
plt.show()

In [15]:
plt.figure(figsize=(12,12))
sns.heatmap(df.isnull(), cbar=False, cmap='Reds')
plt.title('Heatmap Missing Values')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.show()

In [16]:
# tổng số missing toàn dataset
total_missing = df.isnull().sum().sum()
print("Tổng số giá trị thiếu:", total_missing)

# cột nào > 50% missing
high_missing_cols = missing_summary[missing_summary['missing_percent'] > 50]
print("Cột có >50% missing:\n", high_missing_cols.round(2))

In [17]:
# Vẽ biểu đồ cột cho thấy số lượng giá trị bệnh và không bệnh trong target
plt.figure(figsize=(10, 5))

ax = sns.countplot(data=df, x='target', hue='target')

# Gán nhãn cho các cột
for c in ax.containers:
    ax.bar_label(container=c, fontsize=10, fontweight='bold', padding=1)

# Đổi legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, ['No Disease', 'Disease'], title="Target")

# Tiêu đề
ax.set_title("Distribution of Target Variable", fontweight='bold')
ax.set_xlabel("Target")
ax.set_ylabel("Count")

plt.show()

In [18]:
# Tính tỉ lệ phân phối biến mục tiêu
disease_percent = df['target'].value_counts(normalize=True)

# Vẽ biểu đồ tròn thể hiện tỉ lệ phân phối biến mục tiêu
plt.figure(figsize=(5, 5))
plt.pie(disease_percent,
        labels=['No Disease', 'Disease'],
        autopct='%1.1f%%',
        startangle=90)

plt.title("Target Distribution")

plt.show()

In [19]:
# Vẽ các biểu đồ countplot cho danh sách các biến phân loại
fig, axes = plt.subplots(nrows=len(cat_cols), figsize=(10, 5 * len(cat_cols)))

for i, col in enumerate(cat_cols):
    ax = axes[i]
    sns.countplot(data=df, x=col, hue=col, ax=ax, legend=False)

    for c in ax.containers:
        ax.bar_label(container=c, fontsize=10, fontweight='bold', padding=1)

    ax.set_xlabel(col, fontweight='bold')
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [20]:
# Vẽ biểu đồ KDE
fig, axes = plt.subplots(nrows=len(num_cols), ncols=1, figsize=(10, 5 * len(num_cols)))

for i, col in enumerate(num_cols):
        sns.kdeplot(
            data=df,
            x=col,
            hue='target',
            ax=axes[i],
            fill=True,
            alpha=0.2
        )
        axes[i].set_title(col, fontweight='bold')
        axes[i].set_xlabel(col)
plt.tight_layout(pad=3.0)
plt.show()

In [21]:
# Hàm vẽ Boxplot :  Boxplot giúp trực quan hóa phân bố dữ liệu và dễ dàng nhận ra các điểm nằm ngoài phạm vi bình thường.
def plot_boxplot(df, column):
    """
    Vẽ boxplot cho một cột số trong DataFrame.

    Args:
        df (pd.DataFrame): DataFrame chứa dữ liệu đầu vào.
        column (str): Tên cột cần vẽ boxplot.
    """
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[column])

    plt.title(f"Boxplot of {column}")
    plt.xlabel(column)

    plt.show()

In [22]:
# Phát hiện Outlier bằng phương pháp IQR : Phương pháp **Interquartile Range (IQR)** được sử dụng để xác định các giá trị bất thường.
def detect_outlier(df, column):
    """
    Phát hiện outlier trong một cột số bằng phương pháp IQR.

    Args:
        df (pd.DataFrame): DataFrame chứa dữ liệu đầu vào.
        column (str): Tên cột cần kiểm tra outlier.

    Returns:
        pd.DataFrame: Các hàng chứa giá trị nằm ngoài khoảng
            [Q1 - 1.5*IQR, Q3 + 1.5*IQR].
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower) | (df[column] > upper)]

    print(f"\n===== Outlier in {column} =====")
    print("Lower bound:", lower)
    print("Upper bound:", upper)
    print("Number of outliers:", len(outliers))

    return outliers

In [23]:
# Phân tích Outlier cho từng biến:  Vẽ boxplot và Tính số lượng outlier bằng IQR
for col in num_cols:
    plot_boxplot(df, col)
    detect_outlier(df, col)

In [24]:
# Kiểm tra giá trị cực trị (Min và Max): Sau khi phát hiện outlier, chúng ta cần kiểm tra **giá trị nhỏ nhất và lớn nhất** để xác định: Outlier là dữ liệu thật hay lỗi nhập liệu
for col in num_cols:
    print(f"\nColumn: {col}")
    print("Min:", df[col].min())
    print("Max:", df[col].max())

In [25]:
# Encode các categorical feature
df_encoded = df.copy()

for col in cat_cols:
    df_encoded[col] = pd.Categorical(df_encoded[col]).codes

#trong biểu đồ heatmap này các cột id, dataset không có ý nghĩa so sánh và cột num đã được chuyển thành target, nên không cần dùng các cột này
df_encoded = df_encoded.drop(columns=['id', 'dataset', 'num'])

# Tính ma trận tương quan
corr_matrix = df_encoded.corr()

plt.figure(figsize=(20, 10))
sns .heatmap(
    data=corr_matrix,
    annot=True,
    fmt='.2f',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True,
    cmap='RdYlBu_r'
)

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

print('tương quan các cột với target theo độ mạnh:')
print(corr_matrix['target'].drop('target').sort_values(key=abs,ascending=False).round(2))

In [26]:
class OutlierHandler(BaseEstimator, TransformerMixin):
    """Xử lý các giá trị bất hợp lệ về mặt lâm sàng bằng cách thay thế bằng NaN."""
    def fit(self, X, y=None):
        """
        Không học tham số, trả về self để tương thích với Pipeline.

        Args:
            X (pd.DataFrame): Dữ liệu đầu vào.
            y: Bỏ qua, chỉ có mặt để tương thích với scikit-learn API.

        Returns:
            self: Trả về chính transformer.
        """
        # Return self vì bước này không cần học tham số (như mean/std) từ tập train
        return self

    def transform(self, X):
        """Thay thế các giá trị 0 bất khả thi lâm sàng bằng NaN.

        Các cột 'trestbps' (huyết áp) và 'chol' (cholesterol) không thể
        có giá trị 0 trên thực tế, nên 0 được coi là dữ liệu bị thiếu.

        Args:
            X (pd.DataFrame): DataFrame chứa ít nhất hai cột
                'trestbps' và 'chol'.

        Returns:
            pd.DataFrame: Bản sao của X với các giá trị 0 trong
                'trestbps' và 'chol' đã được thay thế bằng NaN.
        """
        X_new = X.copy()

        # 0 là giá trị bất khả thi trên lâm sàng đối với huyết áp và cholesterol
        X_new['trestbps'] = X_new['trestbps'].replace(0, np.nan)
        X_new['chol'] = X_new['chol'].replace(0, np.nan)

        return X_new

In [27]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Tạo các đặc trưng mới (Feature Engineering) dựa trên kiến thức y khoa lâm sàng."""

    def fit(self, X, y=None):
        """
        Không học tham số, trả về self để tương thích với Pipeline.

        Args:
            X (pd.DataFrame): Dữ liệu đầu vào.
            y: Bỏ qua, chỉ có mặt để tương thích với scikit-learn API.

        Returns:
            self: Trả về chính transformer.
        """
        return self

    def transform(self, X):
        """Tạo hai đặc trưng mới: tỷ lệ nhịp tim (heart_rate_ratio) và nhóm rủi ro theo tuổi (age_risk_group).

        - 'heart_rate_ratio': Tỷ lệ giữa nhịp tim thực tế ('thalch') và ngưỡng tối đa lý thuyết (220 - 'age').
        - 'age_risk_group': Phân nhóm độ tuổi thành 4 mức rủi ro (0: 0-40, 1: 40-50, 2: 50-60, 3: 60+)
          dựa trên hiệu ứng ngưỡng nguy hiểm.

        Args:
            X (pd.DataFrame): DataFrame chứa ít nhất hai cột
                'age' và 'thalch'.

        Returns:
            pd.DataFrame: Bản sao của X đã được bổ sung thêm hai cột
                mới là 'heart_rate_ratio' và 'age_risk_group'.
        """
        X_new = X.copy()

        # 1. Feature: heart_rate_ratio
        X_new['heart_rate_ratio'] = X_new['thalch'] / (220 - X_new['age'])

        # 2. Feature: age_risk_group
        # Chia bins: 0–40 (0), 40–50 (1), 50–60 (2), 60+ (3)
        bins = [0, 40, 50, 60, np.inf]
        labels = [0, 1, 2, 3]
        X_new['age_risk_group'] = pd.cut(X_new['age'], bins=bins, labels=labels).astype(int)

        return X_new

In [28]:
#Các cột cần drop theo quyết định ở phần trước
drop_cols = ['id', 'dataset', 'num', 'ca', 'thal', 'target']
X = df.drop(columns=drop_cols)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [29]:
print("Kích thước tập Train:", y_train.shape[0]/y.shape[0] * 100, "%")
print("Kích thước tập Test:", y_test.shape[0]/y.shape[0] * 100, "%")
print()

print("Tỷ lệ class trong y_train:")
print(y_train.value_counts(normalize=True).round(3))
print()
print("Tỷ lệ class trong y_test:")
print(y_test.value_counts(normalize=True).round(3))

In [30]:
num_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'heart_rate_ratio']
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'age_risk_group']

In [31]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ],
    remainder='drop'
)

full_preprocessing_pipeline = Pipeline(steps=[
    ('outlier_handler', OutlierHandler()),
    ('feature_engineer', FeatureEngineer()),
    ('preprocessor', preprocessor)
])

print(full_preprocessing_pipeline)

In [32]:
X_train_processed = full_preprocessing_pipeline.fit_transform(X_train)
X_test_processed = full_preprocessing_pipeline.transform(X_test)

In [33]:
print("Shape X_train trước:", X_train.shape)
print("Shape X_train sau:  ", X_train_processed.shape)
print("Shape X_test trước: ", X_test.shape)
print("Shape X_test sau:   ", X_test_processed.shape)

print()

# Kiểm tra không còn NaN
print("NaN X_train_processed:", np.isnan(X_train_processed).sum())
print("NaN X_test_processed: ", np.isnan(X_test_processed).sum())

print()

# Kiểm tra dtype
print("Dtype:", X_train_processed.dtype)

In [34]:
print(X_train_processed)

In [35]:
fe_transformer = FeatureEngineer()
df_fe = fe_transformer.transform(df)

print("--- VẼ BIỂU ĐỒ 1 (BOXPLOT) ---")
plt.figure(figsize=(10, 6))

sns.boxplot(data=df_fe, x='target', y='heart_rate_ratio', palette='Set2', hue='target', width=0.5)
sns.stripplot(data=df_fe, x='target', y='heart_rate_ratio', color=".3", size=3, alpha=0.5)

plt.title('TỶ LỆ THEO NHÓM NGUY CƠ NHỊP TIM (HEART_RATE_RATIO)', pad=15, fontsize=14, fontweight='bold')
plt.xlabel('Chẩn đoán (0: Khỏe, 1: Bệnh)', fontsize=12)
plt.ylabel('Heart rate ratio', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout(pad=3.0)
plt.show()


print("--- VẼ BIỂU ĐỒ 2 (STACKED BAR) ---")

# Dùng normalize='index' để tính tỷ lệ % nội bộ trong từng nhóm tuổi thay vì trên tổng toàn bộ dataset
data_pct = pd.crosstab(df_fe['age_risk_group'], df_fe['target'], normalize='index') * 100

ax = data_pct.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#66c2a5', '#fc8d62'])

plt.title('TỶ LỆ BỆNH THEO NHÓM NGUY CƠ TUỔI (AGE_RISK_GROUP)', pad=15, fontsize=14, fontweight='bold')
plt.xlabel('Nhóm tuổi (0:[0–40], 1:[40–50], 2:[50–60], 3:[60+])', fontsize=12)
plt.ylabel('Phần trăm (%)', fontsize=12)
plt.xticks(rotation=0)
plt.legend(['Khỏe (Target 0)', 'Bệnh (Target 1)'], loc='upper right', title='Trạng thái')

for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width / 2, y + height / 2, f'{height:.1f}%',
                ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()

In [36]:
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

num_base = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']
cat_base = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope']

preprocessor_base = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_base),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_base)
    ],
    remainder='drop'
)

pipeline_base = Pipeline([
    ('outlier_handler', OutlierHandler()),
    ('preprocessor', preprocessor_base),
    ('model', lr_model)
])

pipeline_fe = Pipeline([
    ('full_preprocessor', full_preprocessing_pipeline),
    ('model', lr_model)
])

f1_base = cross_val_score(pipeline_base, X_train, y_train, cv=cv, scoring='f1').mean()
f1_fe = cross_val_score(pipeline_fe, X_train, y_train, cv=cv, scoring='f1').mean()

print(f"F1 Baseline (Chưa FE): {f1_base:.4f}")
print(f"F1 Sau FE (Có FE): {f1_fe:.4f}")
print(f"Cải thiện: {((f1_fe - f1_base) / f1_base) * 100:+.2f}%")

In [37]:
#Hàm log_model_to_wandb để lưu mô hình và kết quả đánh giá vào WandB
def log_model_to_wandb(model, model_name, config, y_test, y_pred, y_proba, metrics):
    """
    Tạo run, lưu mô hình và kết quả đánh giá vào WandB.
    Args:
        model: mô hình đã được huấn luyện
        model_name: tên mô hình (string)
        config: dictionary chứa các thông số cấu hình của mô hình
        y_test: nhãn thật của tập test
        y_pred: nhãn dự đoán của mô hình trên tập test
        y_proba: xác suất dự đoán của mô hình trên tập test (nếu có)
        metrics: dictionary chứa các chỉ số đánh giá (accuracy, precision, recall, f1_score, roc_auc)

    """

    with wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=model_name,
        config=config,
        reinit=True,
        settings=wandb.Settings(init_timeout=180)
    ) as run:

        # Log metrics
        wandb.log(metrics)

        # Confusion matrix
        wandb.log({
            "confusion_matrix": wandb.plot.confusion_matrix(
                probs=None,
                y_true=list(y_test.reset_index(drop=True)),
                preds=list(y_pred),
                class_names=['No Disease', 'Disease']
            )
        })

        # ROC + PR curve
        if y_proba is not None:
            y_proba_2d = np.column_stack([1 - y_proba, y_proba])

            wandb.log({
                "roc_curve": wandb.plot.roc_curve(
                    y_true=list(y_test.reset_index(drop=True)),
                    y_probas=y_proba_2d,
                    labels=['No Disease', 'Disease']
                ),
                "pr_curve": wandb.plot.pr_curve(
                    y_true=list(y_test.reset_index(drop=True)),
                    y_probas=y_proba_2d,
                    labels=['No Disease', 'Disease']
                )
            })

        # Save model
        model_filename = f"{model_name.replace(' ', '_').lower()}_model.pkl"
        jb.dump(model, model_filename)

        model_artifact = wandb.Artifact(
            model_name.replace(' ', '_').lower() + "_model",
            type='model'
        )
        model_artifact.add_file(model_filename)
        run.log_artifact(model_artifact)

        os.remove(model_filename)

        print(f"Đã log mô hình '{model_name}' và kết quả đánh giá vào WandB thành công!")

In [38]:
# Hàm evaluate_model để đánh giá mô hình, vẽ biểu đồ và trả về số liệu
def evaluate_model(model, model_name, X_test, y_test, threshold=0.5):
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)
        y_prob_positive = y_prob[:, 1]
    else:
        y_prob = None
        y_prob_positive = model.predict(X_test)

    y_pred = (y_prob_positive >= threshold).astype(int)

    test_recall = recall_score(y_test, y_pred)
    test_precision = precision_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_prob_positive)

    print(f"Kết quả Test (Threshold = {threshold}):")
    print(f"   - Recall:    {test_recall:.4f}")
    print(f"   - Precision: {test_precision:.4f}")
    print(f"   - F1-Score:  {test_f1:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Predict 0', 'Predict 1'],
        yticklabels=['Actual 0', 'Actual 1']
    )

    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('Thực tế (Actual)')
    plt.xlabel('Dự đoán (Predicted)')
    plt.tight_layout()
    plt.show()

    metrics_dict = {
        "Test_Recall": test_recall,
        "Test_Precision": test_precision,
        "Test_F1": test_f1,
        "Test_ROC_AUC": test_auc,
        "Threshold_used": threshold
    }

    return metrics_dict, y_pred, y_prob_positive

In [39]:
# Hàm tổng để Train -> Evaluate -> Log lên WandB
def train_and_run_experiment(
    model,
    model_name,
    config,
    X_train,
    y_train,
    X_test,
    y_test,
    threshold=0.5,
    random_state=RANDOM_STATE
):

    print(f"Training {model_name} ...")

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state
    )

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='recall',
        n_jobs=-1
    )

    cv_recall_mean = cv_scores.mean()
    print(f"CV Recall (5-Fold): {cv_recall_mean:.4f}")

    model.fit(X_train, y_train)

    metrics_dict, y_pred, y_prob = evaluate_model(
        model=model,
        model_name=model_name,
        X_test=X_test,
        y_test=y_test,
        threshold=threshold
    )

    print(f"Metrics trên tập Test đã được tính toán: {metrics_dict}")

    metrics_dict["CV_Recall_Mean"] = cv_recall_mean

    log_model_to_wandb(
        model=model,
        model_name=model_name,
        config=config,
        y_test=y_test,
        y_pred=y_pred,
        y_proba=y_prob,
        metrics=metrics_dict
    )

    print(f"Đã log mô hình và kết quả đánh giá của {model_name} lên WandB thành công!")
    print(f"Đã hoàn thành experiment cho {model_name}.\n")

    return model, metrics_dict

In [40]:
#khởi tạo model Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')

In [41]:
#khai báo pram_grid cho GridSearchCV
param_grid_lr = [
    {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']   # liblinear hỗ trợ cả l1 và l2
    },
    {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],        # lbfgs chỉ hỗ trợ l2
        'solver': ['lbfgs']
    }
]

In [42]:
#khởi tạo GridSearchCV cho Logistic Regression với 5-fold CV, scoring là recall,
#sử dụng tất cả CPU có sẵn và verbose để theo dõi tiến trình
grid_lr = GridSearchCV(
    estimator=lr,
    param_grid=param_grid_lr,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

In [43]:
#fit GridSearchCV cho Logistic Regression
grid_lr.fit(X_train_processed, y_train)


print("Best params:", grid_lr.best_params_)
print("Best CV Recall:", grid_lr.best_score_)

In [44]:
#Chạy experiment với model Logistic Regression đã được tối ưu hyperparameters từ GridSearchCV,
#sử dụng threshold 0.4 để đánh giá trên tập Test và log kết quả lên WandB với tên "Logistic Regression"
lr_model, lr_metrics = train_and_run_experiment(
    model=grid_lr.best_estimator_,
    model_name="Logistic Regression",
    config=grid_lr.best_params_,
    X_train=X_train_processed ,
    y_train=y_train,
    X_test=X_test_processed,
    y_test=y_test,
    threshold=THRESHOLD
)

In [45]:
#khởi tạo Random forest
rf_config = {
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_split": 5,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE
}

rf_model = RandomForestClassifier(**rf_config)

In [46]:
#Khai báo param_grid cho Random Forest
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', None]
}

In [47]:
#GridSearchCV Random Forest
rf = RandomForestClassifier(random_state=RANDOM_STATE)

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    scoring="recall",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train_processed, y_train)

print("Best params:", grid_rf.best_params_)
print("Best CV Recall:", grid_rf.best_score_)

In [48]:
#Run 1 (GridSearchCV)
best_params_rf = grid_rf.best_params_

rf_model, rf_metrics = train_and_run_experiment(
    model=RandomForestClassifier(**best_params_rf, random_state=RANDOM_STATE),
    model_name="Random Forest",
    config=best_params_rf,
    X_train=X_train_processed,
    y_train=y_train,
    X_test=X_test_processed,
    y_test=y_test,
    threshold=THRESHOLD
)

In [49]:
#RUN 2: Baseline
rf_baseline = RandomForestClassifier(random_state=RANDOM_STATE)
rf_baseline.fit(X_train_processed, y_train)

# Đánh giá mô hình baseline
metrics_base, y_pred_base, y_proba_base = evaluate_model(
    rf_baseline,
    "Random Forest Baseline",
    X_test_processed,
    y_test,
    threshold=THRESHOLD
)

# Log lên WandB
log_model_to_wandb(
    rf_baseline,
    "Random Forest Baseline",
    rf_baseline.get_params(),
    y_test,
    y_pred_base,
    y_proba_base,
    metrics_base
)

In [50]:
# RUN 2: ExtraConfig
rf_extra = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    class_weight="balanced",
    random_state=RANDOM_STATE
)
rf_extra.fit(X_train_processed, y_train)

# Đánh giá mô hình extra config
metrics_extra, y_pred_extra, y_proba_extra = evaluate_model(
    rf_extra,
    "Random Forest ExtraConfig",
    X_test_processed,
    y_test,
    threshold=THRESHOLD
)

# Log lên WandB
log_model_to_wandb(
    rf_extra,
    "Random Forest ExtraConfig",
    rf_extra.get_params(),
    y_test,
    y_pred_extra,
    y_proba_extra,
    metrics_extra
)

In [51]:
#Cấu hình tham số cho SVM
SVM_param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf']
}

In [52]:
def svm_experiment(SVM_param_grid=None, experiment_suffix="GridSearchCV"):
    """
    Thực hiện quy trình huấn luyện và tối ưu hóa mô hình Support Vector Machine (SVM).

    Hàm này hỗ trợ hai chế độ:
    1. Chế độ GridSearchCV: Nếu SVM_param_grid được cung cấp, hàm sẽ tự động tìm
       bộ tham số (C, gamma, kernel) tối ưu nhất dựa trên chỉ số Recall.
    2. Chế độ Baseline: Nếu không có lưới tham số, hàm sẽ sử dụng cấu hình mặc định.

    Args:
        SVM_param_grid (dict, optional): Từ điển chứa các không gian tham số cần dò tìm.
            Ví dụ: {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']}. Mặc định là None.
        experiment_suffix (str): Hậu tố để đặt tên cho thí nghiệm trên WandB, giúp
            phân biệt giữa các lần chạy (ví dụ: "Baseline", "Tuned").

    Returns:
        tuple:
            - model_trained: Mô hình SVM đã được huấn luyện (phiên bản tốt nhất nếu có GridSearch).
            - metrics (dict): Từ điển chứa các kết quả đánh giá (Accuracy, Recall, Precision, F1).
    """

    # 1. Khởi tạo mô hình cơ bản
    base_model = SVC(probability=True, random_state=RANDOM_STATE)

    # 2. KIỂM TRA: Nếu có truyền lưới tham số, ta thực hiện dò tìm tại đây luôn
    if SVM_param_grid is not None:
        print(f"Đang thực hiện GridSearchCV cho SVM...")
        # Tạo bộ tìm kiếm tập trung vào tối ưu hóa RECALL
        grid_search = GridSearchCV(
            estimator=base_model,
            param_grid=SVM_param_grid,
            scoring='recall',
            cv=5,
            n_jobs=-1
        )
        grid_search.fit(X_train_processed, y_train)

        # Lấy mô hình tốt nhất sau khi đã 'dò' xong
        final_model = grid_search.best_estimator_
        best_params = grid_search.best_params_
        print(f"Bộ tham số tốt nhất tìm được: {best_params}")
    else:
        final_model = base_model
        best_params = "Default"

    # 3. Cấu hình để log lên WandB
    config = {
        "model_type": "SVM",
        "search_space": SVM_param_grid,
        "best_params_found": best_params, # Log luôn bộ số tốt nhất lên WandB
        "notes": "Toi uu hoa Recall cho benh tim"
    }

    # 4.gọi hàm tổng để Train (lần cuối), Evaluate và Log
    exp_name = f"SVM_{experiment_suffix}"

    model_trained, metrics = train_and_run_experiment(
        model=final_model,
        model_name=exp_name,
        config=config,
        X_train=X_train_processed,
        y_train=y_train,
        X_test=X_test_processed,
        y_test=y_test,
        threshold=THRESHOLD
    )

    return model_trained, metrics

In [53]:
# Chạy mô hình SVM thực tế
best_svm, svm_metrics = svm_experiment(SVM_param_grid=SVM_param_grid,experiment_suffix="GridSearchCV")